# ETL — Equidade Territorial de Preços
**TCC — Business Intelligence aplicado ao BPS** · Mônica Anatália — POLI USP PRO

**O que este notebook faz:**
1. Recupera a UF ausente em 2012 a partir do CNPJ da instituição, com validação
2. Deflaciona os preços pelo mês da compra (base dez/2023)
3. Calcula o **índice de preço relativo por UF**: razão entre o preço pago pela
   UF e a mediana nacional do MESMO item no MESMO ano
4. Exporta as tabelas para o Power BI

**Por que índice relativo e não preço médio:** comparar o preço médio entre UFs
seria comparar cestas diferentes — um estado que compra itens caros pareceria
ineficiente. O índice relativo compara cada UF ao preço nacional do mesmo item
no mesmo ano, e depois toma a mediana dessas razões.

## 1. Setup

In [ ]:
# ── CAMINHO DOS DADOS ────────────────────────────────────────────────────
# Ajuste PASTA_DADOS para o local onde estao os arquivos.
# Padrao: subpasta "dados" ao lado do notebook. No Google Colab, aponte para
# a pasta do seu Drive apos monta-lo.
import os
PASTA_DADOS = os.environ.get('BPS_DADOS', 'dados')
# ─────────────────────────────────────────────────────────────────────────
!pip install polars pyarrow --quiet
import polars as pl
from pathlib import Path

BASE        = Path(PASTA_DADOS)
PASTA_BPS   = BASE / 'Base Harmonizacao Campos'
ARQ_IPCA    = BASE / 'IPCA Tratado' / 'ipca_deflator.parquet'
PASTA_SAIDA = BASE / 'Base Territorial'
ANO_INICIO, ANO_FIM = 2009, 2023
MIN_UFS, MIN_REG = 5, 500   # item comprado por >=5 UFs; UF com >=500 registros
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
print('Pronto ✓')

## 2. Carregar

In [ ]:
COLS=['data_compra','preco_unitario','valor_total','cod_catmat','unidade_chave',
      'uf','cnpj_instituicao','modalidade_compra']
fr=[]
for ano in range(2000, 2026):
    arq = PASTA_BPS / f'BPS_{ano}.parquet'
    if not arq.exists(): continue
    disp = pl.read_parquet(arq, n_rows=0).columns
    fr.append(pl.read_parquet(arq, columns=[c for c in COLS if c in disp]))
bruto = pl.concat(fr, how='diagonal')
print(f'Registros lidos: {bruto.height:,}')

def norm_cnpj(col):
    return pl.col(col).cast(pl.Utf8).str.replace_all(r'\D','').str.pad_start(14,'0')
def uf_ok(col):
    return pl.col(col).cast(pl.Utf8).str.strip_chars().str.len_chars() > 0

## 3. Recuperação da UF ausente (2012)
> No arquivo-fonte de 2012 as colunas UF, Município e Nome da Instituição estão
> preenchidas em apenas 40,6% dos registros — o CNPJ da instituição, porém, está
> em 100%. Como a mesma instituição aparece em outros anos com UF informada,
> a UF é recuperada por correspondência de CNPJ.
>
> **Validação:** o metodo e aplicado aos registros em que a UF ORIGINAL existe e
> o resultado e comparado ao valor real. A celula falha se o acerto nao for total.

In [ ]:
mapa = (bruto.filter(uf_ok('uf'))
    .select(norm_cnpj('cnpj_instituicao').alias('cnpj'),
            pl.col('uf').cast(pl.Utf8).str.strip_chars().alias('u'))
    .group_by(['cnpj','u']).agg(pl.len().alias('n')).sort('n', descending=True)
    .group_by('cnpj').agg(pl.col('u').first().alias('uf_rec')))

amb = (bruto.filter(uf_ok('uf'))
        .select(norm_cnpj('cnpj_instituicao').alias('cnpj'),
                pl.col('uf').cast(pl.Utf8).str.strip_chars().alias('u')).unique()
        .group_by('cnpj').agg(pl.col('u').n_unique().alias('k')))
print(f'CNPJs mapeados: {mapa.height:,} | com UF ambigua: {amb.filter(pl.col("k")>1).height:,}')

base = bruto.with_columns(norm_cnpj('cnpj_instituicao').alias('cnpj')).join(mapa, on='cnpj', how='left')

# validacao contra os casos em que a UF original existe
val = base.filter(uf_ok('uf') & pl.col('uf_rec').is_not_null())
acerto = val.filter(pl.col('uf').cast(pl.Utf8).str.strip_chars() == pl.col('uf_rec')).height / val.height
print(f'VALIDACAO: acerto de {acerto:.2%} em {val.height:,} casos de controle')
assert acerto > 0.999, 'Recuperacao de UF por CNPJ nao confiavel'

base = base.with_columns(
    pl.coalesce([pl.when(uf_ok('uf')).then(pl.col('uf').cast(pl.Utf8).str.strip_chars()),
                 pl.col('uf_rec')]).alias('uf_f'))
print(f'Cobertura de UF apos recuperacao: {base.filter(pl.col("uf_f").is_not_null()).height/base.height:.1%}')

## 4. Deflacionar e aplicar o recorte

In [ ]:
ipca = pl.read_parquet(ARQ_IPCA).select(['ano','mes','fator_deflacao'])
b = (base
     .filter(pl.col('data_compra').is_not_null()
             & pl.col('data_compra').dt.year().is_between(ANO_INICIO, ANO_FIM)
             & (pl.col('preco_unitario') > 0)
             & pl.col('cod_catmat').is_not_null())
     .with_columns(pl.col('data_compra').dt.year().alias('ano'),
                   pl.col('data_compra').dt.month().alias('mes'))
     .join(ipca, on=['ano','mes'], how='left')
     .with_columns((pl.col('preco_unitario')*pl.col('fator_deflacao')).alias('preco_real')))
assert b.filter(pl.col('fator_deflacao').is_null()).height == 0
print(f'Registros na analise: {b.height:,}')

## 5. Índice de preço relativo por UF
> Para cada item-ano calcula-se a mediana nacional. So entram itens comprados por
> pelo menos MIN_UFS unidades federativas, para que a comparacao seja significativa.
> O indice da UF e a MEDIANA das razoes preco_UF / mediana_nacional.

In [ ]:
it = b.filter(pl.col('uf_f').is_not_null()).with_columns(
        (pl.col('cod_catmat') + '|' + pl.col('unidade_chave').fill_null('')).alias('item'))

nac = it.group_by(['ano','item']).agg(pl.col('preco_real').median().alias('p_nac'),
                                      pl.col('uf_f').n_unique().alias('n_uf'))
rel = (it.join(nac, on=['ano','item'])
         .filter(pl.col('n_uf') >= MIN_UFS)
         .with_columns((pl.col('preco_real')/pl.col('p_nac')).alias('rel')))
print(f'Base do indice: {rel.height:,} registros')

indice_uf = (rel.group_by('uf_f').agg(pl.col('rel').median().round(3).alias('indice'),
                                      pl.len().alias('registros'),
                                      pl.col('item').n_unique().alias('itens'))
               .filter(pl.col('registros') >= MIN_REG)
               .sort('indice').rename({'uf_f':'uf'}))
print(indice_uf)
print(f'\nAmplitude: {indice_uf["indice"].min():.3f} a {indice_uf["indice"].max():.3f}')

## 6. Série por UF e ano (para o painel)

In [ ]:
indice_uf_ano = (rel.group_by(['ano','uf_f']).agg(pl.col('rel').median().round(3).alias('indice'),
                                                  pl.len().alias('registros'))
                   .filter(pl.col('registros') >= 50)
                   .sort(['ano','uf_f']).rename({'uf_f':'uf'}))
print(indice_uf_ano.head(10))

## 7. Exportar

In [ ]:
for nome, df in [('indice_territorial_uf', indice_uf),
                 ('indice_territorial_uf_ano', indice_uf_ano)]:
    df.write_parquet(PASTA_SAIDA / f'{nome}.parquet')
    df.write_csv(PASTA_SAIDA / f'{nome}.csv')
print('=== EXPORTADO ===')
for f in sorted(PASTA_SAIDA.iterdir()): print(' ', f.name)
print('\nValores de referencia para conferencia:')
print('  cobertura de UF apos recuperacao: 100,0%')
print('  validacao da recuperacao        : 100,00%')
print('  base do indice                  : 692.482 registros')
print('  menor indice: SC 0,910 | maior: PI 1,813')

## Memória de cálculo — recuperação da UF e índice territorial
> Imprime a conta de cada valor citado no texto: a validacao da recuperacao,
> a cobertura antes e depois, e a construcao do indice de preco relativo.

In [ ]:
print('='*72); print('1. RECUPERACAO DA UF EM 2012'); print('='*72)
d12 = base.filter(pl.col('data_compra').dt.year() == 2012)
com_uf  = d12.filter(uf_ok('uf')).height
sem_uf  = d12.height - com_uf
recup   = d12.filter(~uf_ok('uf') & pl.col('uf_rec').is_not_null()).height
print(f'  registros de 2012                 : {d12.height:>7,}')
print(f'  com UF na fonte                   : {com_uf:>7,} = {com_uf/d12.height:.1%}')
print(f'  sem UF na fonte                   : {sem_uf:>7,}')
print(f'  recuperados pelo CNPJ             : {recup:>7,} = {recup/sem_uf:.1%} dos ausentes')
print(f'  cobertura final                   : {(com_uf+recup)/d12.height:.1%}')
print(f'\n  CNPJs mapeados                    : {mapa.height:>7,}')
print(f'  com UF ambigua                    : {amb.filter(pl.col("k")>1).height:>7,}')
print('\n  VALIDACAO (metodo aplicado a registros que JA tinham UF):')
ctrl = d12.filter(uf_ok("uf") & pl.col("uf_rec").is_not_null())
ac = ctrl.filter(pl.col('uf').cast(pl.Utf8).str.strip_chars() == pl.col('uf_rec')).height
print(f'    casos de controle               : {ctrl.height:>7,}')
print(f'    acertos                         : {ac:>7,}  ->  {ac/ctrl.height:.2%}')

In [ ]:
print('='*72); print('2. INDICE DE PRECO RELATIVO POR UF'); print('='*72)
print('  Para cada item-ano: mediana nacional do preco real.')
print('  Para cada registro: razao preco / mediana nacional.')
print('  Indice da UF     : MEDIANA dessas razoes.')
print(f'\n  itens-ano com >= {MIN_UFS} UFs compradoras : {nac.filter(pl.col("n_uf")>=MIN_UFS).height:,}')
print(f'  registros na base do indice        : {rel.height:,}')
print(f'  UFs com >= {MIN_REG} registros        : {indice_uf.height}')
print(f'\n{"UF":>4} | {"indice":>7} | {"leitura":<34} | {"registros":>9}')
for r in indice_uf.iter_rows(named=True):
    dif = (r['indice']-1)*100
    txt = f'paga {abs(dif):.1f}% {"abaixo" if dif<0 else "acima"} da referencia'
    print(f'{r["uf"]:>4} | {r["indice"]:>7.3f} | {txt:<34} | {r["registros"]:>9,}')
amp = indice_uf['indice'].max()/indice_uf['indice'].min()
print(f'\n  amplitude: {indice_uf["indice"].max():.3f} / {indice_uf["indice"].min():.3f} = {amp:.2f}x  ->  diferenca de {(amp-1)*100:.0f}%')